## Məlumatların Yüklənməsi

In [ ]:
!kaggle competitions download -c ctrl-is-hers-final

In [ ]:
!unzip /kaggle/working/ctrl-is-hers-final.zip

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import seaborn as sns
import shap
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              ExtraTreesClassifier, AdaBoostClassifier)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import time

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## Məlumatların oxunması və ilkin baxış

#### `train.csv` və `test.csv` fayllarının yüklənməsi, ölçülərinin və data tiplərinin yoxlanılması

In [ ]:
df = pd.read_csv('train.csv')
pd.set_option('display.max_columns', None)
df.head()

In [ ]:
test_df = pd.read_csv('test.csv')
test_ids = test_df['id']
print(df.shape, test_df.shape)

In [ ]:
df.info()

## Tarix xüsusiyyətlərinin (features) hazırlanması

In [ ]:
def add_date_features(data):
    data = data.copy()
    data['publish_date'] = pd.to_datetime(data['publish_date'])
    data['publish_year'] = data['publish_date'].dt.year
    data['publish_month'] = data['publish_date'].dt.month
    data['publish_day'] = data['publish_date'].dt.day
    data['publish_hour'] = data['publish_date'].dt.hour
    return data.drop(columns=['publish_date'])

## Boşluq və unikal dəyərlərin təhlili

In [ ]:
pd.DataFrame({
    "boşluq %": (df.isnull().mean() * 100).round(1),
    "unikal":   df.nunique(),
    "nümunə":   [df[c].dropna().unique()[:5] for c in df.columns],
})

In [ ]:
df = add_date_features(df)
test_df = add_date_features(test_df)

## Lazımsız sütunların silinməsi

#### `id` və `weekday` sütunları çıxarılır; test üçün `id` ayrıca saxlanılır.

In [ ]:
test_ids = test_df['id']
df = df.drop(columns=['id', 'weekday'], errors='ignore')
X_test_final = test_df.drop(columns=['id', 'weekday'], errors='ignore')

In [ ]:
X = df.drop(columns='target', axis=1)
y = df['target']

### Train/Val bölgüsü

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2,
                                                   random_state=42,
                                                   stratify=y)

In [ ]:
X_train.shape, X_val.shape

## Preprocessing

In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns
cat_cols = X_train.select_dtypes(exclude=[np.number]).columns

In [ ]:
num_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('scaler', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

preprocess = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
], remainder='passthrough')

In [ ]:
X_train_preprocessed = preprocess.fit_transform(X_train)
X_val_preprocessed = preprocess.transform(X_val)

In [ ]:
X_test_final_prep = preprocess.transform(X_test_final)

## Baseline model - Logistic Regression

In [ ]:
base_model = LogisticRegression(max_iter=1000)
base_model.fit(X_train_preprocessed, y_train)

In [ ]:
base_model.score(X_train_preprocessed, y_train)

In [ ]:
base_model.score(X_val_preprocessed, y_val)

### Validation üçün ehtimalların hesablanması

In [ ]:
y_proba = base_model.predict_proba(X_val_preprocessed)[:, 1]
y_proba

In [ ]:
test_proba = base_model.predict_proba(X_test_final_prep)[:, 1]

In [ ]:
len(test_ids)

In [ ]:
len(test_proba)

In [ ]:
submission = pd.DataFrame({
    'id': test_ids,
    'target': test_proba
})

## ROC-AUC qiymətləndirməsi

In [ ]:
# Test roc-auc score
roc_auc_score(y_val, y_proba)

In [ ]:
submission.to_csv('sample2.csv', index=False)